In [1]:
# Import libraries
import pandas as pd
from sqlalchemy import create_engine
import urllib
from dotenv import load_dotenv
import os

In [2]:
# Connecting to Azure SQL
load_dotenv()

conn = urllib.parse.quote_plus(
    f"DRIVER=ODBC Driver 17 for SQL Server;"
    f"SERVER={os.getenv('DB_SERVER')};"
    f"DATABASE={os.getenv('DB_NAME')};"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')}"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={conn}")
print("Connected to Azure SQL!")

Connected to Azure SQL!


1. Write a function that takes a price and returns its tier. Test it on 5 different prices.

In [3]:
def price_to_tier(prices):
    for p in prices:
        if p < 5:
            print("Budget")
        elif p> 5 and p < 25:
            print("Basic")
        elif p > 25 and p < 75:
            print("Standard")
        elif p > 75 and p < 150:
            print("Premium")
        else:
            print("Luxury")

prices = [15, 89, 199, 350, 1200]
price_to_tier(prices)

Basic
Premium
Luxury
Luxury
Luxury


2. Create a list of 5 Brazilian states. Loop through and print each with its position number.

In [4]:
brasilian_states = ["Sao Paulo", "Rio de Janeiro", "Minas Gerais", "Bahia", "Parana"]

for i in range(len(brasilian_states)):
    print(i, ":", brasilian_states[i])

0 : Sao Paulo
1 : Rio de Janeiro
2 : Minas Gerais
3 : Bahia
4 : Parana


4. Run a query returning all delivered orders from 2018. Print the row count.

In [6]:
df = pd.read_sql("""
    Select ood.order_id
    From olist_orders_dataset ood 
    Where ood.order_status = 'delivered' 
        And ood.order_delivered_customer_date >= '2018-01-01'
        And ood.order_delivered_customer_date < '2019-01-01'
""", engine)

print(f"Rows returned: {len(df)}")

Rows returned: 55273


5. Run the top 10 most expensive items query. Store in a dataframe and print it.

In [8]:
df = pd.read_sql("""
    Select Top 10 ooid.order_id,
        ooid.price 
    From olist_order_items_dataset ooid 
    Order By ooid.price Desc
""", engine)

print(df)

                           order_id        price
0  0812eb902a67711a1cb742b3cdaa65ae  6735.000000
1  fefacc66af859508bf1a7934eab1e97f  6729.000000
2  f5136e38d1a14a4dbd87dff67da82701  6499.000000
3  a96610ab360d42a2e5335a3998b4718a  4799.000000
4  199af31afc78c699f0dbf71fb178d4d4  4690.000000
5  8dbc85d1447242f3b127dda390d56e19  4590.000000
6  426a9742b533fc6fed17d1fd6d143d7e  4399.870117
7  68101694e5c5dc7330c91e1bbc36214f  4099.990234
8  b239ca7cd485940b31882363b52e6674  4059.000000
9  86c4eab1571921a6a6e248ed312f5a5a  3999.899902


6. Write a function that takes a state code and returns all customers from that state.

In [13]:
def customers_on_state(state_code, engine):
    df = pd.read_sql(f"""
        Select ocd.customer_id 
        From olist_customers_dataset ocd
        Where ocd.customer_state = '{state_code}'
    """, engine)

    print(df)

state_code = "SP"
customers_on_state(state_code, engine)

                            customer_id
0      06b8999e2fba1a1fbc88172c00ba8bc7
1      18955e83d337fd6b2def6b18a428ac77
2      4e7b3e00288586ebd08712fdd0374a03
3      b2b6027bc5c5109e529d4dc6358b12c3
4      4f2d8ab171c80ec8364f7c12e35b23ad
...                                 ...
41741  f255d679c7c86c24ef4861320d5b7675
41742  f5a0b560f9e9427792a88bec97710212
41743  17ddf5dd5d51696bb3d7c6291687be6f
41744  e7b71a9017aa05c9a7fd292d714858e8
41745  274fa6071e5e17fe303b9748641082c8

[41746 rows x 1 columns]


7. Build a dynamic query using an f-string that filters by order status variable.

In [14]:
order_status = 'shipped'

df = pd.read_sql(f"""
    Select ood.order_id,
	    ood.order_status 
    From olist_orders_dataset ood 
    Where ood.order_status = '{order_status}'
""", engine)

print(df)

                              order_id order_status
0     ee64d42b8cf066f35eac1cf57de1aa85      shipped
1     6942b8da583c2f9957e990d028607019      shipped
2     36530871a5e80138db53bcfd8a104d90      shipped
3     4d630f57194f5aba1a3d12ce23e71cd9      shipped
4     3b4ad687e7e5190db827e1ae5a8989dd      shipped
...                                ...          ...
1102  a59ef0abffbef8ddaae23600b6ee6604      shipped
1103  dab8a6c6bd6ec448df5b3a6b6cb887bc      shipped
1104  492aed3c33bac22a8e04138319829283      shipped
1105  274a7f7e4f1c17b7434a830e9b8759b1      shipped
1106  636cdd02667dc8d76d9296bf20a6890a      shipped

[1107 rows x 2 columns]


8. Run the GROUP BY query from Week 1 through Python. Print the result.

In [15]:
df = pd.read_sql("""
    Select ocd.customer_state,
        Count(ocd.customer_id) As orders_per_state
    From olist_customers_dataset ocd 
    Group By ocd.customer_state 
    Order By orders_per_state Desc
""", engine)

df.head(5)

,customer_state,orders_per_state
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045


9. Load the full orders table into pandas. Print shape, columns, and null counts.

In [23]:
df = pd.read_sql("""
    Select * 
    From olist_orders_dataset ood 
""", engine)

print("Columns: \n", df.columns)
print('Shape: ', df.shape)
print('Null_values: \n', df.isnull().sum())

Columns: 
 Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
Shape:  (99441, 8)
Null_values: 
 order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64
